# 01 - Canonical Data Validation and Readiness (POC-41)
Read-only notebook. Source: `data/canonical/intelligence_data.csv` only. No charts, features or models. Run from this folder after `python scripts/data_pipeline/run_pipeline.py` and the four `data-science/scripts` programs.

In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().resolve().parents[1]
df = pd.read_csv(ROOT/'data'/'canonical'/'intelligence_data.csv', dtype={'is_synthetic': str})
man = json.loads((ROOT/'data'/'manifest.json').read_text())
out = lambda n: json.loads((ROOT/'data-science'/'outputs'/n).read_text())
print('rows x cols:', df.shape, '| data_version:', sorted(df.data_version.unique()), '| manifest:', man['data_version'])

rows x cols: (129, 20) | data_version: ['1.0.0'] | manifest: 1.0.0


## 1. Structural gate (Post #1 validation, re-run)

In [2]:
q = out('quality_assessment.json')
print(q['post1_validation'])
print('Quality status:', q['status'], '| blocking issues:', [i for i in q['issues'] if i['severity']=='blocking'])

{'command': 'python scripts/data_pipeline/validate_data.py', 'result': 'PASS', 'summary': 'Validation PASS (39/39 checks)'}
Quality status: PASS | blocking issues: []


## 2. Record types, sources and provenance

In [3]:
print(df.record_type.value_counts().to_string()); print()
print(df.groupby(['source_name','is_synthetic']).size().to_string())

record_type
measurement    123
entity           6

source_name                                           is_synthetic
Backend synthetic shift generator (main.py, seed 42)  true            63
Frontend mockDatabase (page.tsx)                      true            66


## 3. Field completeness

In [4]:
c = df.isna().sum().to_frame('null'); c['null_pct'] = (c['null']/len(df)*100).round(1); c['unique'] = df.nunique()
print(c.to_string())

                   null  null_pct  unique
record_id             0       0.0     129
record_type           0       0.0       2
observed_at           6       4.7       7
entity_id             0       0.0       9
related_entity_id   129     100.0       0
entity_name           0       0.0       9
category              0       0.0       3
subcategory          63      48.8       2
status                0       0.0       3
stage               129     100.0       0
metric_name           0       0.0       4
metric_value          0       0.0      99
metric_unit           0       0.0       3
text_value          123      95.3       6
latitude            129     100.0       0
longitude           129     100.0       0
source_name           0       0.0       2
source_record_id      0       0.0     129
is_synthetic          0       0.0       1
data_version          0       0.0       1


## 4. Numerical coverage by source and metric

In [5]:
m = df[df.metric_value.notna()].groupby(['source_name','metric_name','metric_unit']).metric_value.agg(['count','min','max','mean','median','std']).round(2)
print(m.to_string())

                                                                                         count     min     max    mean  median    std
source_name                                          metric_name         metric_unit                                                 
Backend synthetic shift generator (main.py, seed 42) attendance_rate     ratio              21    0.85    0.98    0.91    0.90   0.04
                                                     output_per_hour     units_per_hour     21  111.90  149.50  129.65  131.00  13.23
                                                     overtime_hours      hours              21    5.70   38.90   20.44   17.50  11.06
Frontend mockDatabase (page.tsx)                     kpi_output_per_hour units_per_hour      6   95.20  142.10  121.58  123.45  17.22
                                                     output_per_hour     units_per_hour     30   85.00  142.00  119.40  121.00  15.21
                                                     overtime_

## 5. Temporal coverage (observations per series)

In [6]:
d = df[df.observed_at.notna()].copy(); d['day'] = d.observed_at.str[:10]
print(d.groupby('source_name').day.agg(['min','max','nunique']).to_string()); print()
print(d.groupby(['source_name','entity_id','metric_name']).size().groupby('source_name').agg(['count','min','max']).rename(columns={'count':'series'}).to_string())

                                                             min         max  nunique
source_name                                                                          
Backend synthetic shift generator (main.py, seed 42)  2026-10-24  2026-10-30        7
Frontend mockDatabase (page.tsx)                      2026-10-24  2026-10-28        5

                                                      series  min  max
source_name                                                           
Backend synthetic shift generator (main.py, seed 42)       9    7    7
Frontend mockDatabase (page.tsx)                          12    5    5


## 6. Categories, status, geography, text

In [7]:
for col in ['category','subcategory','status']:
    print(col, df[col].fillna('<missing>').value_counts().to_dict())
print('coordinates populated:', int(df.latitude.notna().sum()), '| stage populated:', int(df.stage.notna().sum()))
print('text values:', int(df.text_value.notna().sum()), 'unique:', int(df.text_value.nunique()))

category {'evening_crew': 43, 'morning_crew': 43, 'night_watch': 43}
subcategory {'<missing>': 63, 'facility_a': 33, 'facility_b': 33}
status {'optimal': 61, 'warning': 48, 'critical': 20}
coordinates populated: 0 | stage populated: 0
text values: 6 unique: 6


## 7. Quality observations (non-blocking)

In [8]:
for o in q['observations']:
    print(o['id'], '-', o['finding'], '\n   ', o['detail'])

OBS-1 - Headline KPI is not derivable from the daily series 
    kpi_output_per_hour differs from the latest daily output by up to 4.1 units and from the 5-day mean by up to 5.1; treat KPI as a separate source-stated value.
OBS-2 - Frontend status is monotone in the KPI 
    Statuses ordered by KPI are consistent: True. Thresholds are implicit in the UI code, not documented.
OBS-3 - Backend status is not tied to backend metrics 
    Backend coverage_status is drawn at random (70/20/10) independently of output; mean output by status: {"critical": 137.97, "optimal": 127.15, "warning": 131.18}. It is a label, not an outcome derived from the metrics.
OBS-4 - Different scopes share metric names 
    output_per_hour and overtime_hours exist in both sources (facility-level vs shift-level). Always filter by source_name; do not pool.
OBS-5 - Short, fully synthetic series 
    Frontend series have 5 daily points, backend 7. All rows are synthetic.


## 8. Representativeness

In [9]:
r = out('representativeness_assessment.json')
print(r['result'], '-', r['scope_of_result']); print(r['original_sampling']); print(r['underrepresented_areas'])

PASS - PASS applies to the 7-day comparative view of 2 facilities x 3 teams. It does NOT cover 30-day or year-to-date horizons.
{'method': 'full_population_retained', 'original_records': 51, 'sample_records': 51, 'canonical_records': 129, 'full_population_retained': True}
['Time: only 5-7 consecutive days; no weekly cycle, month or year coverage.', 'Backend rows have no facility, so facility comparison exists only in the frontend source.', 'Critical cases are rare: 1 frontend node and ~3 backend shift-days.']


## 9. Analytical readiness matrix

In [10]:
a = out('analytical_readiness.json')
print(pd.DataFrame(a['tracks'])[['track','status']].to_string(index=False)); print()
print('Primary question:', a['primary_question']['question']); print('Predictive:', a['predictive_intelligence']['status']); print('FINAL:', a['final_result'])

              track      status
    A - Comparative   SUPPORTED
          B - Trend CONDITIONAL
C - Risk & Priority CONDITIONAL
        D - Anomaly    REJECTED
   E - Segmentation    REJECTED
     F - Predictive    REJECTED
     G - Simulation    REJECTED
   H - Text & Theme    REJECTED

Primary question: Across Facility A and B, which team's output per hour and overtime hours differ most from the other groups during 2026-10-24 to 2026-10-28, and how consistent is that gap day to day?
Predictive: REJECTED
FINAL: DATA READY FOR ANALYTICAL TRACK DEVELOPMENT
